In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [13]:
import pandas as pd
query = pd.read_csv('queries_rag.csv')['query']

In [14]:
query

,query
0,Give me a conversation between two people disc...
1,Give me a conversation between two people disc...
2,Give me a conversation between two people disc...
3,Give me a conversation between two people disc...
4,Give me a conversation between two people disc...
5,Give me a conversation between two people disc...
6,Give me a conversation between two people disc...
7,Give me a conversation between two people disc...
8,Give me a conversation between two people disc...
9,Give me a conversation between two people disc...


### Fine-tune model

In [4]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# 從 Hugging Face 載入模型
config = PeftConfig.from_pretrained("coconut19/llama-dialog-lora")
base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(base_model, "coconut19/llama-dialog-lora")
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


adapter_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [5]:
import torch
import tqdm
model.eval()

def generate_dialog(user_instruction, max_new_tokens=512):
    prompt = f"Instruction: {user_instruction}\nAnswer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_length=50,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

lora_model_output = []
for q in tqdm.tqdm(query):
    generated_text = generate_dialog(q, max_new_tokens=100)
    lora_model_output.append(generated_text)

100%|██████████| 10/10 [07:35<00:00, 45.59s/it]


In [6]:
df = pd.DataFrame({
    'query': query,
    'lora_model_output': lora_model_output
})
df.to_csv('rag_evaluation_results.csv', index=False)

### Fine-tune + RAG

In [15]:
retrieved_context = pd.read_csv('retrieved_results.csv')['content']
retrieved_context

,content
0,The Poisson distribution can be derived as an ...
1,"In C++, a class is a user-defined data type th..."
2,"In C++, a file is treated as a sequence of byt..."
3,C++ is a general-purpose programming language ...
4,The cumulative distribution function (CDF) of ...
5,Newton’s First Law states that an object remai...
6,A random variable is a function that maps each...
7,The Central Limit Theorem states that the sum ...
8,Dynamic memory allocation in C allows programs...
9,A pointer is a variable that stores the memory...


In [17]:
model.eval()

def generate_dialog(user_instruction, retrieved_context, max_new_tokens=256):
    prompt = f"""
    Use the information provided in the relevant context to generate the answer.
    ### Relevant Context:
    {retrieved_context}
    ### Instruction:Instruction: {user_instruction}\n ### Answer:\n"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

lora_model_rag_output = []
for q, ctx in tqdm.tqdm(zip(query, retrieved_context), total=len(query)):
    generated_text = generate_dialog(q, ctx, max_new_tokens=150)
    lora_model_rag_output.append(generated_text)

100%|██████████| 10/10 [09:48<00:00, 58.88s/it]


In [18]:
df = pd.DataFrame({
    'query': query,
    'lora_model_output': lora_model_rag_output
})
df.to_csv('rag_evaluation_results_rag.csv', index=False)